<a href="https://colab.research.google.com/github/IsaacVic-Dark/Gemma-FineTune-colab-notebooks/blob/main/Download_InsuOps_Model_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
import os
!pip uninstall -y torchao unsloth unsloth_zoo peft bitsandbytes accelerate xformers trl cut_cross_entropy

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
ADAPTER_DIR = "/content/drive/MyDrive/Insuops_lora_model_v2"
print("Adapter files:", os.listdir(ADAPTER_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model']


In [3]:
from unsloth import FastModel
from peft import PeftModel

BASE_MODEL_NAME = "unsloth/gemma-3-1b-it"

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    dtype=None,
    load_in_4bit=False,
    load_in_8bit=False,
    max_seq_length=2048,
)

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model = model.merge_and_unload()
print("✅ Merged. Type:", type(model))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✅ Merged. Type: <class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForCausalLM'>


In [4]:
import shutil

HF_MERGED_DIR = "/content/Inc_hf"
if os.path.exists(HF_MERGED_DIR):
    shutil.rmtree(HF_MERGED_DIR)
os.makedirs(HF_MERGED_DIR)

model.save_pretrained(HF_MERGED_DIR)
tokenizer.save_pretrained(HF_MERGED_DIR)

files = os.listdir(HF_MERGED_DIR)
print("Files:", files)
assert "config.json" in files, "❌ Merge failed — config.json missing!"
print("✅ Merge confirmed")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in /content/Inc_hf/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/Inc_hf.


Files: ['tokenizer_config.json', 'model.safetensors', 'generation_config.json', 'tokenizer.model', 'chat_template.jinja', 'config.json', 'tokenizer.json']
✅ Merge confirmed


In [5]:
from unsloth import FastModel

# Reload cleanly from the merged folder
model_gguf, tokenizer_gguf = FastModel.from_pretrained(
    model_name=HF_MERGED_DIR,
    dtype=None,
    load_in_4bit=False,
    max_seq_length=2048,
)

GGUF_OUTPUT_DIR = "/content/Inc_gguf"
os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)

model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method="q4_k_m"
)

print("GGUF files:", os.listdir(GGUF_OUTPUT_DIR))

==((====))==  Unsloth 2026.5.7: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Model is not a PEFT model. Using existing checkpoint at /content/Inc_hf
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/Inc_hf_gguf/Inc_hf.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/Inc_hf_gguf/Inc_hf.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model '/content/Inc_hf'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/Inc_hf_gguf/Inc_hf.Q4_K_M.gguf -p "why is the sky blue?"
GGUF files: ['tokenizer_config.json', 'adapter_model.safetensors', 'generation_config.json', 'adapter_config.json', 'tokenizer.model', 'chat_template.jinja', 'tokenizer.json']


In [6]:
# This zips and triggers a browser download to your Windows machine
import shutil
from google.colab import files

# The GGUF file will be named something like model-q4_k_m.gguf
gguf_files = [f for f in os.listdir(GGUF_OUTPUT_DIR) if f.endswith(".gguf")]
print("GGUF files found:", gguf_files)

for f in gguf_files:
    files.download(os.path.join(GGUF_OUTPUT_DIR, f))

GGUF files found: []


In [8]:
from google.colab import files

files.download("/content/Inc_hf_gguf/Inc_hf.Q4_K_M.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>